#### **Tools**

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

- A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
- A function or coroutine to execute.

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(model="groq:openai/gpt-oss-120b")
response = model.invoke("Why is New Yok so busy?")
response


AIMessage(content='New\u202fYork City feels like a nonstop whirlwind for a few key reasons:\n\n| Factor | What it means for the city | Why it adds to the “busy” vibe |\n|--------|---------------------------|--------------------------------|\n| **Population density** | Over 8\u202fmillion people live in the five boroughs, and the metro area tops 20\u202fmillion. | More people → more commuters, shoppers, diners, and pedestrians everywhere you look. |\n| **Economic hub** | Home to Wall\u202fStreet, major media companies, tech startups, fashion houses, and countless corporate headquarters. | Millions travel in and out each day for work, meetings, and networking events. |\n| **Tourism magnet** | Attractions like Times\u202fSquare, Central\u202fPark, the Statue of Liberty, Broadway, museums, and world‑class restaurants draw ~65\u202fmillion visitors annually. | Tourists flood streets, subways, and taxis, especially in the summer and around holidays. |\n| **Transportation network** | One of t

In [ ]:
from langchain.tools import tool

### Create a own tool function and bind it to the model
@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "What\'s the weather like in Boston?" We have a function get_weather. We should call it with location "Boston".', 'tool_calls': [{'id': 'fc_a71b6fbc-3ee7-42a2-81de-448ec19a88f8', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 127, 'total_tokens': 183, 'completion_time': 0.116658058, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.004810852, 'prompt_tokens_details': None, 'queue_time': 0.22146738, 'total_time': 0.12146891}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_27194a498a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a123b6-9b45-7983-9952-ee5059496632-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_a71b6fbc-3ee7-42a2-81de-448ec19a88f8', 'type'

#### **Tool Execution Loops**

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It’s sunny in Boston right now.


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What\'s the weather in Boston?" Need to call get_weather function with location "Boston".', 'tool_calls': [{'id': 'fc_bbf28241-5bee-4b60-9632-293122713e53', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 126, 'total_tokens': 175, 'completion_time': 0.102321748, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.004749069, 'prompt_tokens_details': None, 'queue_time': 0.091702054, 'total_time': 0.107070817}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a123b7-5f1e-7ab1-8fbf-2830a5cac518-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 